# Transformers

### Neil D. Lawrence

### 2025-10-01

**Abstract**: This lecture builds on deep neural networks to explore
transformer architectures, focusing on how attention mechanisms require
sophisticated chain rule applications and how they connect to the
overparameterization and generalization themes.

$$
$$

<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## ML Foundations Course Notebook Setup

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install some bespoke codes for creating and saving plots as well as
loading data sets.

In [ ]:
%%capture
%pip install notutils
%pip install git+https://github.com/lawrennd/ods.git
%pip install git+https://github.com/lawrennd/mlai.git

In [ ]:
import notutils
import pods
import mlai
import mlai.plot as plot

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

# From Deep Networks to Transformers

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/transformers-and-llms.gpp.markdown" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/transformers-and-llms.gpp.markdown', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In our previous lectures, we explored how composing layers of basis
functions creates deep neural networks, and we examined the chain rule
and automatic differentiation that makes training these networks
possible. We’ve seen how we can consider structured data through
convolutional neural networks, graph neural networks, recurrent
networks. Today we’ll see how these foundations extend to one of the
most important architectural innovations in deep learning: the
transformer.

Transformers represent a fundamental shift from the sequential
processing of RNNs to parallel attention mechanisms. This creates new
challenges for automatic differentiation, as we’ll see.

# The Attention Mechanism

The key insight of transformers is the attention mechanism, which allows
the model to focus on different parts of the input sequence
simultaneously. This creates a more complex gradient flow than standard
neural networks.

## Chain Rule for Transformer Attention

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/chain-rule-transformer-attention.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/chain-rule-transformer-attention.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

The transformer attention mechanism is more complex than standard neural
networks because the same input matrix $\mathbf{X}$ appears in three
different linear transformations. This creates a more intricate chain
rule when computing gradients.

This multiple appearance is what allows the transformer to include
variable length sequences. But it make sthe chain rule computation a
little more complex than for a standard neural network.

The attention mechanism computes a weighted combination of values, where
the weights are determined by the similarity between queries and keys.
The softmax ensures the weights sum to one.

In a standard neural network, we have a single path from input to
output. In transformer attention, we have three parallel paths through
the same input, making the chain rule more complex.

The gradient flow through attention involves computing how the loss
changes with respect to the attention weights and the value matrix.

The gradient with respect to the attention matrix comes from the product
with the value matrix. This tells us how much each attention weight
should change.

The gradient through the softmax requires the standard softmax gradient
formula, accounting for the fact that attention weights sum to one.

The gradients for queries and keys come from their interaction in the
attention logits. Each query interacts with all keys, and each key
interacts with all queries.

Finally, we combine all three gradient paths to get the gradient with
respect to the input matrix. This is the key insight: the same input
appears in three different transformations.

The gradients for the weight matrices follow the standard pattern: input
matrix transposed times the gradient of the output.

Multi-head attention adds another layer of complexity. Each head
computes its own attention, and the gradients must be computed for each
head separately before being combined.

Implementing transformer gradients efficiently requires careful
attention to memory usage and numerical stability. The softmax operation
can be numerically unstable for large attention scores.

The transformer attention mechanism requires a more sophisticated
understanding of the chain rule because the same input participates in
multiple parallel computations.

# Transformer Architecture

Now we’ll see how to build a complete transformer model, integrating all
the components we’ve discussed.

## Simple Transformer Implementation

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/simple-transformer-implementation.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/simple-transformer-implementation.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import numpy as np

In [ ]:
def create_synthetic_sequence_data(n_samples=100, seq_length=10, vocab_size=50):
    """Create synthetic sequence data for demonstration."""
    np.random.seed(24)
    
    # Create random sequences
    X = np.random.randint(0, vocab_size, (n_samples, seq_length))
    
    # Create target: next token prediction
    y = np.roll(X, -1, axis=1)
    y[:, -1] = 0  # Padding token for last position
    
    return X, y

In [ ]:
# Create data
X_seq, y_seq = create_synthetic_sequence_data(200, 8, 30)

print(f"Sequence data: {X_seq.shape} -> {y_seq.shape}")
print(f"Sample sequence: {X_seq[0]}")
print(f"Target sequence: {y_seq[0]}")

## Create and Test Basic Attention

In [ ]:
from mlai import Attention
import inspect
file_path = inspect.getfile(Attention)

In [ ]:
%load -s Attention {file_path}

In [ ]:
d_model = 64
n_heads = 4
seq_length = 8
vocab_size = 30

# Create basic attention mechanism
attention = Attention(d_model)

# Test forward pass
X_test = np.random.randn(2, seq_length, d_model)
attn_output, attn_weights = attention.forward(X_test, X_test, X_test)

print("Basic Attention Test:")
print(f"Input shape: {X_test.shape}")
print(f"Output shape: {attn_output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention weights sum (should be 1): {attn_weights.sum(axis=-1)[0, 0]}")
print(f"Model parameters: {attention.W_q.size + attention.W_k.size + attention.W_v.size + attention.W_o.size}")

## Test Multi-Head Attention

In [ ]:
from mlai import MultiHeadAttention
import inspect
file_path = inspect.getfile(MultiHeadAttention)

In [ ]:
%load -s MultiHeadAttention {file_path}

In [ ]:
# Test multi-head attention (built from basic attention)
multi_head_attention = MultiHeadAttention(d_model, n_heads)

# Forward pass
X_test = np.random.randn(2, seq_length, d_model)
attn_output, attn_weights = multi_head_attention.forward(X_test, X_test, X_test)

print("Multi-Head Attention Test:")
print(f"Input shape: {X_test.shape}")
print(f"Output shape: {attn_output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention weights sum (should be 1): {attn_weights.sum(axis=-1)[0, 0]}")
print(f"Number of heads: {n_heads}")

## Test Chain Rule in Attention

In [ ]:
# Test gradient flow through attention (demonstrating chain rule)
from mlai import MeanSquaredError

X_test = np.random.randn(2, seq_length, d_model)

# Forward pass through attention
output, attn_weights = attention.forward(X_test, X_test, X_test)

# Create dummy loss using proper loss function (consistent with neural network)
target = np.random.randn(2, seq_length, d_model)
loss_fn = MeanSquaredError()
loss_value = loss_fn.forward(output, target)

# Backward pass (demonstrates three-path chain rule)
loss_gradient = loss_fn.gradient(output, target)
gradients = attention.backward(loss_gradient, X_test, X_test, X_test, attn_weights)

print("Chain Rule Demonstration:")
print(f"Loss value: {loss_value:.4f}")
print(f"Input gradient shape: {gradients['grad_input'].shape}")
print(f"Input gradient norm: {np.linalg.norm(gradients['grad_input']):.4f}")
print("This shows how gradients flow through Q, K, V transformations")
print(f"Three-path chain rule: grad_query + grad_key + grad_value = grad_input")
print(f"Gradient verification: {np.allclose(gradients['grad_input'], gradients['grad_query'] + gradients['grad_key'] + gradients['grad_value'])}")

## Train Simple Attention Model

In [ ]:
def train_attention_model(X, y, n_epochs=50, learning_rate=0.001):
    """Train a simple attention model for sequence modeling."""
    from mlai import MeanSquaredError
    
    # Create model
    d_model = 64
    n_heads = 4
    vocab_size = 30
    
    # Use multi-head attention for learning
    model = MultiHeadAttention(d_model, n_heads)
    
    # Convert to embeddings (simplified)
    X_embedded = np.random.randn(len(X), X.shape[1], d_model)
    y_embedded = np.random.randn(len(y), y.shape[1], d_model)
    
    # Loss function (consistent with neural network)
    loss_fn = MeanSquaredError()
    
    # Training loop
    losses = []
    for epoch in range(n_epochs):
        # Forward pass through attention
        output, attn_weights = model.forward(X_embedded, X_embedded, X_embedded)
        
        # Compute loss using proper loss function
        loss = loss_fn.forward(output, y_embedded)
        
        # Backward pass (demonstrates chain rule)
        loss_gradient = loss_fn.gradient(output, y_embedded)
        
        # Use the existing backward method from the Attention class
        # This demonstrates proper gradient computation through the attention mechanism
        for i, head in enumerate(model.attention_heads):
            # Get the corresponding head data
            head_query = X_embedded[:, :, i*model.d_k:(i+1)*model.d_k]
            head_key = X_embedded[:, :, i*model.d_k:(i+1)*model.d_k] 
            head_value = X_embedded[:, :, i*model.d_k:(i+1)*model.d_k]
            head_weights = attn_weights[:, i]  # Get weights for this head
            
            # Compute gradients using the existing backward method
            gradients = head.backward(loss_gradient[:, :, i*model.d_k:(i+1)*model.d_k], 
                                    head_query, head_key, head_value, head_weights)
            
            # Update weights using computed gradients
            head.W_q -= learning_rate * gradients['grad_W_q']
            head.W_k -= learning_rate * gradients['grad_W_k']
            head.W_v -= learning_rate * gradients['grad_W_v']
            head.W_o -= learning_rate * gradients['grad_W_o']
        
        losses.append(loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}: Loss = {loss:.4f}")
    
    return model, losses

In [ ]:
X_seq, y_seq = create_synthetic_sequence_data(200, 8, 30)
model, losses = train_attention_model(X_seq, y_seq)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot

In [ ]:
fig, ax = plt.subplots(figsize=plot.wide_figsize)
ax.plot(losses, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
plt.grid(True, alpha=0.3)

print(f"Final training loss: {losses[-1]:.4f}")

mlai.write_figure("transformer-training-progress.svg", directory="./deepnn")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//deepnn/transformer-training-progress.svg" class="" width="50%" style="vertical-align:middle;">

Figure: <i>Transformer Training Progress for sequence modeling</i>

## Visualise Attention Weights

In [ ]:
def visualise_attention_weights(model, X, filename="attention-weights.svg", directory="../diagrams"):
    """Visualise attention weights from the multi-head attention model."""
    # Get attention weights
    X_embedded = np.random.randn(1, X.shape[1], model.d_model)
    _, attn_weights = model.forward(X_embedded, X_embedded, X_embedded)
    
    # Get first head, first sample
    attn_weights = attn_weights[0, 0]  # First head, first sample
    
    # Plot
    fig, ax = plt.subplots(figsize=plot.big_figsize)
    im = ax.imshow(attn_weights, cmap='Blues', aspect='auto')
    
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    
    mlai.write_figure(filename, directory=directory)

In [ ]:
visualise_attention_weights(model, X_seq, "attention-weights.svg", directory="./deepnn/")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//deepnn/attention-weights.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Attention weights visualisation from the first head showing
which positions the model attends to</i>

## Test Different Numbers of Heads

In [ ]:
# Test different numbers of heads (showing composition)
d_model = 64
seq_length = 8

for n_heads in [1, 2, 4, 8]:
    multi_head_attention = MultiHeadAttention(d_model, n_heads)
    X_test = np.random.randn(2, seq_length, d_model)
    
    output, attn_weights = multi_head_attention.forward(X_test, X_test, X_test)
    
    print(f"n_heads={n_heads}: output shape={output.shape}, attn shape={attn_weights.shape}")
    print(f"  Each head processes {d_model//n_heads} dimensions")

## Test Different Activation Functions

In [ ]:
from mlai import SoftmaxActivation
import inspect
file_path = inspect.getfile(SoftmaxActivation)

In [ ]:
%load -s SoftmaxActivation {file_path}

In [ ]:
from mlai import SigmoidAttentionActivation
import inspect
file_path = inspect.getfile(SigmoidAttentionActivation)

In [ ]:
%load -s SigmoidAttentionActivation {file_path}

In [ ]:
from mlai import IdentityMinusSoftmaxActivation
import inspect
file_path = inspect.getfile(IdentityMinusSoftmaxActivation)

In [ ]:
%load -s IdentityMinusSoftmaxActivation {file_path}

In [ ]:
# Compare different activation functions for attention
from mlai import SoftmaxActivation, SigmoidAttentionActivation, IdentityMinusSoftmaxActivation

d_model = 32
seq_length = 4
X_test = np.random.randn(1, seq_length, d_model)

print("Comparing Attention Activation Functions:")
print("=" * 50)

# Standard softmax attention
softmax_attention = Attention(d_model, activation=SoftmaxActivation())
output_softmax, weights_softmax = softmax_attention.forward(X_test, X_test, X_test)

print("1. SoftmaxActivation (Standard):")
print(f"   Weights sum: {weights_softmax.sum(axis=-1)[0, 0]:.6f}")
print(f"   Weights range: [{weights_softmax.min():.6f}, {weights_softmax.max():.6f}]")
print(f"   Attention matrix:\n{weights_softmax[0, :, :]}")

# Sigmoid with normalization
sigmoid_attention = Attention(d_model, activation=SigmoidAttentionActivation())
output_sigmoid, weights_sigmoid = sigmoid_attention.forward(X_test, X_test, X_test)

print("\n2. SigmoidAttentionActivation:")
print(f"   Weights sum: {weights_sigmoid.sum(axis=-1)[0, 0]:.6f}")
print(f"   Weights range: [{weights_sigmoid.min():.6f}, {weights_sigmoid.max():.6f}]")
print(f"   Attention matrix:\n{weights_sigmoid[0, :, :]}")

# Identity minus softmax (interesting alternative)
identity_attention = Attention(d_model, activation=IdentityMinusSoftmaxActivation())
output_identity, weights_identity = identity_attention.forward(X_test, X_test, X_test)

print("\n3. IdentityMinusSoftmaxActivation:")
print(f"   Weights sum: {weights_identity.sum(axis=-1)[0, 0]:.6f}")
print(f"   Weights range: [{weights_identity.min():.6f}, {weights_identity.max():.6f}]")
print(f"   Diagonal entries (1-softmax): {np.diag(weights_identity[0, :, :])}")
print(f"   Off-diagonal entries (-softmax): {weights_identity[0, 0, 1]:.6f}, {weights_identity[0, 1, 0]:.6f}")
print(f"   Attention matrix:\n{weights_identity[0, :, :]}")

print("\nKey Differences:")
print("- Softmax: Standard attention, weights sum to 1, all positive")
print("- Sigmoid: Alternative activation, weights sum to 1, all positive") 
print("- Identity-Minus-Softmax: Diagonal positive (1-softmax), off-diagonal negative (-softmax), sum to 0")
print("  This creates a 'contrast' attention pattern that emphasizes self-connections while")
print("  de-emphasizing connections to other positions.")

## Visualise Different Attention Patterns

In [ ]:
def visualise_attention_patterns():
    """Visualise different attention activation patterns."""
    import matplotlib.pyplot as plt
    
    d_model = 16
    seq_length = 6
    X_test = np.random.randn(1, seq_length, d_model)
    
    # Create attention mechanisms with different activations
    activations = [
        ('Softmax', SoftmaxActivation()),
        ('Sigmoid+Norm', SigmoidAttentionActivation()),
        ('Identity-Softmax', IdentityMinusSoftmaxActivation())
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for i, (name, activation) in enumerate(activations):
        attention = Attention(d_model, activation=activation)
        _, weights = attention.forward(X_test, X_test, X_test)
        
        # Get attention matrix for first sample
        attn_matrix = weights[0, 0, :, :]
        
        # Plot
        im = axes[i].imshow(attn_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        axes[i].set_title(f'{name}\nSum: {attn_matrix.sum():.3f}')
        axes[i].set_xlabel('Key Position')
        axes[i].set_ylabel('Query Position')
        
        # Add colorbar
        plt.colorbar(im, ax=axes[i])
        
        # Add text annotations for small matrices
        if seq_length <= 6:
            for row in range(seq_length):
                for col in range(seq_length):
                    text = axes[i].text(col, row, f'{attn_matrix[row, col]:.2f}',
                                      ha="center", va="center", color="black", fontsize=8)
    
    plt.tight_layout()
    return fig

In [ ]:
fig = visualise_attention_patterns()
mlai.write_figure("attention-activation-comparison.svg", directory="./deepnn/")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//deepnn/attention-activation-comparison.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Comparison of different attention activation functions
showing how they create different attention patterns</i>

In [ ]:
# Show the different attention patterns
print("Attention Pattern Analysis:")
print("Softmax: Standard attention, weights sum to 1")
print("Sigmoid: Alternative activation, weights sum to 1") 
print("Identity-Softmax: Contrast attention, weights sum to 0")
print("This demonstrates different attention behaviors!")

The different attention activation functions create fundamentally
different behaviors:

**Softmax Attention (Standard):** - All weights are positive (0 to 1) -
Each row sums to 1 (probability distribution) - Represents ‘how much to
attend to each position’ - Higher values = more attention

**Sigmoid + Normalization:** - Similar to softmax but uses sigmoid
activation - All weights positive, rows sum to 1 - Alternative way to
create attention weights

**Identity Minus Softmax:** - Diagonal entries: positive (1 - softmax) -
Off-diagonal entries: negative (-softmax) - Each row sums to 0 (not 1!)
- Creates ‘contrast’ attention: \* Positive diagonal = ‘attend to self’
\* Negative off-diagonal = ‘de-emphasize others’ - Could be useful for
tasks requiring: \* Self-focus (diagonal emphasis) \* Contrast learning
(positive vs negative weights) \* Sparse attention patterns

This demonstrates how different activation functions can create
fundamentally different attention behaviors, even with the same
underlying Q, K, V computation!

## Positional Encoding Test

In [ ]:
from mlai import PositionalEncoding
import inspect
file_path = inspect.getfile(PositionalEncoding)

In [ ]:
%load -s PositionalEncoding {file_path}

In [ ]:
# Test positional encoding
pe = PositionalEncoding(d_model, max_length=100)
X_test = np.random.randn(2, seq_length, d_model)

X_with_pe = pe.forward(X_test)

print("Positional Encoding Test:")
print(f"Input shape: {X_test.shape}")
print(f"Output shape: {X_with_pe.shape}")
print(f"PE added: {np.allclose(X_test + pe.pe[:seq_length], X_with_pe)}")

## Simple Transformer Model

In [ ]:
from mlai import Transformer
import inspect
file_path = inspect.getfile(Transformer)

In [ ]:
%load -s Transformer {file_path}

In [ ]:
# Create transformer model using the proper Model class
vocab_size = 30
d_model = 64
n_heads = 4

# Use the Transformer model class (inherits from Model)
transformer = Transformer(d_model=d_model, n_heads=n_heads, vocab_size=vocab_size)

print("Transformer Model Test:")
print(f"Model dimension: {transformer.d_model}")
print(f"Number of heads: {transformer.n_heads}")
print(f"Vocabulary size: {transformer.vocab_size}")
print(f"Is a Model: {isinstance(transformer, Model)}")
print(f"Has predict method: {hasattr(transformer, 'predict')}")
print(f"Has objective method: {hasattr(transformer, 'objective')}")
print(f"Has fit method: {hasattr(transformer, 'fit')}")

In [ ]:
# Test forward pass with proper Model interface
X_test = np.random.randint(0, vocab_size, (2, 8))
output = transformer.predict(X_test)

print("Transformer Model Test:")
print(f"Input shape: {X_test.shape}")
print(f"Output shape: {output.shape}")
print(f"Model parameters: {transformer.embedding.size + transformer.output_projection.size + sum(head.W_q.size + head.W_k.size + head.W_v.size + head.W_o.size for head in transformer.attention.attention_heads)}")
print("This model follows the same pattern as NeuralNetwork - it's a proper Model class")

## Training Simple Model

In [ ]:
def train_transformer_model(X, y, vocab_size, d_model=64, n_heads=4, n_epochs=100, learning_rate=0.001):
    """Train the transformer model using the proper Model class."""
    from mlai import MeanSquaredError
    
    # Create transformer model (inherits from Model)
    model = Transformer(d_model=d_model, n_heads=n_heads, vocab_size=vocab_size)
    
    # Loss function (consistent with neural network)
    loss_fn = MeanSquaredError()
    
    losses = []
    for epoch in range(n_epochs):
        # Forward pass using predict method (Model interface)
        output = model.predict(X)
        
        # Create dummy target for demonstration
        target = np.random.randn(*output.shape)
        
        # Compute loss using proper loss function
        loss = loss_fn.forward(output, target)
        
        # Compute loss gradient using proper loss function
        loss_gradient = loss_fn.gradient(output, target)
        
        # Simple gradient descent update (for demonstration)
        # In practice, you'd implement proper backpropagation
        model.embedding -= learning_rate * np.mean(loss_gradient, axis=(0, 1), keepdims=True)
        model.output_projection -= learning_rate * np.mean(loss_gradient, axis=(0, 1), keepdims=True)
        
        losses.append(loss)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d}: Loss = {loss:.4f}")
    
    return model, losses

In [ ]:
X_seq, y_seq = create_synthetic_sequence_data(200, 8, 30)
transformer_model, transformer_losses = train_transformer_model(X_seq, y_seq, vocab_size=30)

In [ ]:
fig, ax = plt.subplots(figsize=plot.wide_figsize)
ax.plot(transformer_losses, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Transformer Model Training Progress')
plt.grid(True, alpha=0.3)

print(f"Final training loss: {transformer_losses[-1]:.4f}")

mlai.write_figure(“simple-transformer-training.svg,”
directory=“./deepnn/”)}

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//deepnn/simple-transformer-training.svg" class="" width="50%" style="vertical-align:middle;">

Figure: <i>Simple Transformer Training Progress</i>

# Training and Generalization

How do transformers relate to the overparameterization and
generalization themes we discussed in the previous lecture?

## Attention as Implicit Regularization

The attention mechanism provides a form of implicit regularization.
Unlike the explicit regularization we discussed for standard neural
networks, attention creates sparse, interpretable patterns that emerge
during training.

## Overparameterization in Transformers

Transformers generalize well precisely because they are highly
overparameterized. This extends our previous discussion of how
overparameterization enables generalization through the optimization
process, with the attention mechanism providing additional structural
constraints.

# Summary and Future Directions

Transformers represent a significant evolution in deep learning
architectures, but they also raise new questions about optimization,
generalization, and the fundamental principles of learning. The
attention mechanism provides a new form of inductive bias that we’re
still learning to understand theoretically.

## Further Reading

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

::: {.cell .markdown}

## References